# T2: LoRA fine-tune of Qwen2.5-1.5B-Instruct for prescription to JSON

**Run on Kaggle** (Settings: Accelerator GPU T4, Internet on) **or Colab** (T4). About 60 to 90 minutes.

**What this is.** The model reads a typed prescription and returns the JSON MedSaarthi's extractor expects. It is fine-tuned on 20,000 *synthetic* prescriptions built from medicine names. It is not trained on real prescriptions, and the result must be described that way.

**Before and after.** The same 200 held-out prescriptions (medicine names the model never trained on) are scored before and after training, so the improvement is measured, not assumed. Nothing here has been run on the author's laptop (no GPU); paste the printed table into `docs/EVAL.md` only after you run it.

**Steps.** 1) In the repo run `python scripts/make_lora_data.py`, then upload `data/synth/t2_train.jsonl` and `t2_eval.jsonl` as a Kaggle dataset (or to Colab's file panel). 2) Run all cells. 3) Download `t2_results.json` and `t2_adapter.zip`.

In [ ]:
!pip -q install -U transformers peft accelerate datasets

In [ ]:
import glob, json, os, re, time, random
import torch
from datasets import Dataset

def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(f'/content/**/{name}', recursive=True) + glob.glob(name)
    assert hits, f'upload {name} first'
    return hits[0]

TRAIN, EVAL = find('t2_train.jsonl'), find('t2_eval.jsonl')
rows = lambda p: [json.loads(l) for l in open(p, encoding='utf-8')]
train_rows, eval_rows = rows(TRAIN), rows(EVAL)[:200]
print(len(train_rows), 'train,', len(eval_rows), 'held-out eval')
print(train_rows[0]['messages'][1]['content'], '\n->', train_rows[0]['messages'][2]['content'])

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = 'left'
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map={'': 0})
print(sum(p.numel() for p in model.parameters()) / 1e9, 'B parameters')

In [ ]:
FIELDS = ['name', 'dose', 'route', 'frequency', 'duration', 'instructions']
norm = lambda v: ' '.join(str(v).lower().split()) if v not in (None, '') else None

def parse(text):
    m = re.search(r'\{.*\}', text, re.S)
    try:
        d = json.loads(m.group()) if m else None
        return d['medications'] if isinstance(d, dict) and isinstance(d.get('medications'), list) else None
    except Exception:
        return None

@torch.no_grad()
def evaluate(model, rows, batch=16):
    model.eval(); ok_json = fields_ok = fields_all = items_ok = 0
    for i in range(0, len(rows), batch):
        chunk = rows[i:i + batch]
        prompts = [tok.apply_chat_template(r['messages'][:2], tokenize=False, add_generation_prompt=True) for r in chunk]
        enc = tok(prompts, return_tensors='pt', padding=True).to(model.device)
        out = model.generate(**enc, max_new_tokens=320, do_sample=False, pad_token_id=tok.eos_token_id)
        for r, o in zip(chunk, out):
            pred = parse(tok.decode(o[enc.input_ids.shape[1]:], skip_special_tokens=True))
            gold = json.loads(r['messages'][2]['content'])['medications']
            ok_json += pred is not None
            pred = pred or []
            items_ok += len(pred) == len(gold)
            for j, g in enumerate(gold):
                p = pred[j] if j < len(pred) and isinstance(pred[j], dict) else {}
                for f in FIELDS:
                    fields_all += 1
                    fields_ok += norm(p.get(f)) == norm(g.get(f))
    n = len(rows)
    return {'valid_json_pct': round(100 * ok_json / n, 1), 'item_count_match_pct': round(100 * items_ok / n, 1),
            'field_exact_pct': round(100 * fields_ok / fields_all, 1), 'n_prescriptions': n, 'n_fields': fields_all}

t = time.time(); before = evaluate(model, eval_rows); print('BEFORE', before, round(time.time() - t), 's')

In [ ]:
from peft import LoraConfig, get_peft_model
model.gradient_checkpointing_enable(); model.enable_input_require_grads()
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']))
for p in model.parameters():
    if p.requires_grad: p.data = p.data.float()   # fp32 adapters under fp16 autocast
model.print_trainable_parameters()

def encode(r):
    prompt = tok.apply_chat_template(r['messages'][:2], tokenize=False, add_generation_prompt=True)
    p = tok(prompt, add_special_tokens=False).input_ids
    a = tok(r['messages'][2]['content'] + '<|im_end|>', add_special_tokens=False).input_ids
    ids, labels = (p + a)[:700], ([-100] * len(p) + a)[:700]   # loss only on the answer
    return {'input_ids': ids, 'attention_mask': [1] * len(ids), 'labels': labels}

ds = Dataset.from_list(train_rows).map(lambda r: encode(r), remove_columns=['messages'])
print(len(ds), 'examples, longest', max(len(x) for x in ds['input_ids']), 'tokens')

In [ ]:
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments
tok.padding_side = 'right'
args = TrainingArguments(output_dir='t2_out', per_device_train_batch_size=8, gradient_accumulation_steps=4,
    num_train_epochs=1, learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.03, fp16=True,
    logging_steps=25, save_strategy='no', report_to='none', gradient_checkpointing=True, remove_unused_columns=False)
trainer = Trainer(model=model, args=args, train_dataset=ds,
    data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100))
t = time.time(); trainer.train(); train_s = round(time.time() - t); print('trained in', train_s, 's')

In [ ]:
tok.padding_side = 'left'; model.gradient_checkpointing_disable(); model.config.use_cache = True
after = evaluate(model, eval_rows); print('AFTER', after)
results = {'model': MODEL, 'method': 'LoRA r=16', 'train_examples': len(train_rows), 'epochs': 1, 'train_seconds': train_s,
           'eval': 'held-out medicine names, synthetic', 'before': before, 'after': after}
json.dump(results, open('t2_results.json', 'w'), indent=1)
model.save_pretrained('t2_adapter'); tok.save_pretrained('t2_adapter')
!zip -qr t2_adapter.zip t2_adapter
print(json.dumps(results, indent=1))

## Reading the result

Held-out names mean the model cannot just remember a medicine. Synthetic prescriptions are cleaner than real ones, so a high score here is an upper bound, not a hospital accuracy. Compare against the prompt-only LLM in `docs/EVAL.md` (25 hand-written prescriptions) and state both numbers with their data sets.